# 9.1 Save Load Model

訓練一個 Keras 模型，儲存成 `.keras`，再重新載入並推論。

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

## 1. 載入資料

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
x_train = x_train[..., np.newaxis]
x_test = x_test[..., np.newaxis]

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
print(x_train.shape, x_test.shape)

## 2. 建立並訓練模型

In [ ]:
tf.keras.utils.set_random_seed(42)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28, 1)),
    tf.keras.layers.Conv2D(32, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history = model.fit(x_train, y_train, validation_split=0.1, epochs=3, batch_size=128)

## 3. 儲存模型

In [ ]:
model_path = 'fashion_mnist_cnn.keras'
model.save(model_path)
print('model saved:', model_path)
print('file exists:', os.path.exists(model_path))

## 4. 載入模型

In [ ]:
loaded_model = tf.keras.models.load_model(model_path)
loaded_model.summary()

## 5. 比較原模型與載入模型

In [ ]:
original_result = model.evaluate(x_test, y_test, verbose=0, return_dict=True)
loaded_result = loaded_model.evaluate(x_test, y_test, verbose=0, return_dict=True)

pd.DataFrame({
    'metric': list(original_result.keys()),
    'original_model': list(original_result.values()),
    'loaded_model': list(loaded_result.values())
})

## 6. 使用載入模型推論

In [ ]:
sample = x_test[:5]
prob = loaded_model.predict(sample, verbose=0)
pred = np.argmax(prob, axis=1)

pd.DataFrame({
    'predicted_label': pred,
    'predicted_class': [class_names[i] for i in pred],
    'confidence': np.max(prob, axis=1),
    'true_label': y_test[:5],
    'true_class': [class_names[i] for i in y_test[:5]]
})

## 7. 套用自己的模型

若前處理步驟已經放在 Keras 模型中，保存 `.keras` 檔會更容易部署。若前處理使用外部工具，例如 scikit-learn scaler，請另外保存該物件。